In [36]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Use TensorFlow 2.x Keras API
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import regularizers

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


In [37]:
# Load the cleaned fighter data and match data
# Note: You'll need to download the match data from Kaggle or create it
try:
    df_fighters = pd.read_csv("fighters_clean.csv")
    print(f"Loaded {len(df_fighters)} fighters")
except FileNotFoundError:
    print("Error: fighters_clean.csv not found. Please run UFC_Cleaning.ipynb first.")
    df_fighters = pd.DataFrame()

try:
    df_matches = pd.read_csv("matches_clean.csv")
    print(f"Loaded {len(df_matches)} matches")
except FileNotFoundError:
    print("Warning: matches_clean.csv not found.")
    print("You'll need to download the UFC match data from Kaggle:")
    print("https://www.kaggle.com/rajeevw/ufcdata")
    print("Or create matches_clean.csv with columns: Red, Blue, Winner, Weight Class")
    df_matches = pd.DataFrame()

Loaded 4444 fighters
Loaded 8263 matches


In [38]:
df_fighters.set_index("Name", inplace=True)
df_fighters.head()

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage
Name,,,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40


In [39]:
df_fighters['Total Fights'] = df_fighters['Wins'] + df_fighters['Losses'] + df_fighters['Draws']
df_fighters

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Wins,Losses,Draws,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,5,3,0,0.62,8
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,4,6,0,0.40,10
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,28,4,0,0.88,32
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,9,5,0,0.64,14
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,10,15,0,0.40,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Alex Zuniga,No Name,145,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,6,3,0,0.67,9
George Zuniga,No Name,185,NaN,7.64,0.38,5.45,0.37,0.00,0.00,1.00,0.0,"5' 9""",0,3,1,0,0.75,4
Allan Zuniga,Tigre,155,33.8,3.93,0.52,1.80,0.61,0.00,0.00,0.57,1.0,"5' 7""",70,13,1,0,0.93,14


In [40]:
# Drop columns that we don't need for training (only if they exist)
cols_to_drop = ["Weight Class", "Wins", "Losses", "Draws"]
cols_to_drop = [col for col in cols_to_drop if col in df_fighters.columns]
if cols_to_drop:
    df_fighters.drop(cols_to_drop, axis=1, inplace=True)
    print(f"Dropped columns: {cols_to_drop}")
else:
    print("No columns to drop (they may have already been removed)")


Dropped columns: ['Wins', 'Losses', 'Draws']


In [41]:
df_fighters.head(1)

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0,0.62,8


In [42]:
# Get only numeric columns for stats (exclude Name, Nickname, and other non-numeric columns)
stats = [col for col in df_fighters.columns if df_fighters[col].dtype in ['int64', 'float64']]
# Also include Height: if it exists (will convert to numeric later if needed)
if 'Height:' in df_fighters.columns and 'Height:' not in stats:
    stats.append('Height:')
stats

['Weight:',
 'Age',
 'SLpM:',
 'Str. Acc.:',
 'SApM:',
 'Str. Def:',
 'TD Avg.:',
 'TD Acc.:',
 'TD Def.:',
 'Sub. Avg.:',
 'Reach:',
 'Win Percentage',
 'Total Fights',
 'Height:']

In [43]:
df_matches.head(4)

,Red,Blue,Winner,Weight Class
0,Brandon Royval,Manel Kape,Manel Kape,Flyweight
1,Giga Chikadze,Kevin Vallejos,Kevin Vallejos,Featherweight
2,Cesar Almeida,Cezary Oleksiejczuk,Cezary Oleksiejczuk,Middleweight
3,Melquizael Costa,Morgan Charriere,Melquizael Costa,Featherweight


In [44]:
# Drop Weight Class column if it exists
if "Weight Class" in df_matches.columns:
    df_matches.drop("Weight Class", axis=1, inplace=True)
    print("Dropped 'Weight Class' column from matches")
else:
    print("'Weight Class' column not found in matches (may have already been removed)")

Dropped 'Weight Class' column from matches


In [45]:
df_matches["result"] = (df_matches["Winner"] == df_matches["Red"]).astype("int")
df_matches.drop("Winner", axis=1, inplace=True)

In [46]:
df_matches.head()

# if df_matches.loc[df_matches['result']==1:
#     for i =10 to 570
#         df_red = 
    
# idx = (df_matches['result'] == 1)

,Red,Blue,result
0,Brandon Royval,Manel Kape,0
1,Giga Chikadze,Kevin Vallejos,0
2,Cesar Almeida,Cezary Oleksiejczuk,0
3,Melquizael Costa,Morgan Charriere,1
4,Kennedy Nzechukwu,Marcus Buchecha,0


In [47]:
# make sure all fighters are in the figher database and remove any that aren't
fighters_list = df_fighters.index.values.tolist()

df_matches = df_matches.loc[(df_matches["Red"].isin(fighters_list)) &
                                  (df_matches["Blue"].isin(fighters_list))].copy()

df_fighters.head()

,Nickname,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Height:,Reach:,Win Percentage,Total Fights
Name,,,,,,,,,,,,,,,
Tom Aaron,No Name,155,47.5,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,NaN,0,0.62,8
Danny Abbadi,The Assassin,155,42.5,3.29,0.38,4.41,0.57,0.00,0.00,0.77,0.0,"5' 11""",0,0.40,10
Nariman Abbasov,Bayraktar,155,31.9,3.00,0.20,5.67,0.46,0.00,0.00,0.66,0.0,"5' 8""",66,0.88,32
Darion Abbey,No Name,265,32.9,8.44,0.50,14.06,0.28,0.00,0.00,0.00,0.0,"6' 2""",80,0.64,14
David Abbott,Tank,265,60.7,1.35,0.30,3.55,0.38,1.07,0.33,0.66,0.0,"6' 0""",0,0.40,25


In [66]:
# turn df_matches into fighter 1 - fighter 2 (the differences) for training the model
# First, ensure we only use numeric columns and convert Height: to numeric if needed
numeric_stats = []

# Helper function to convert height string to inches
def height_to_inches(height_str):
    """Convert height string like '5\' 11"' to total inches (71.0)"""
    if pd.isna(height_str) or height_str == '':
        return np.nan
    try:
        # Format: "5' 11"" or "5'11""
        parts = str(height_str).replace('"', '').split("'")
        if len(parts) == 2:
            feet = float(parts[0].strip())
            inches = float(parts[1].strip())
            return feet * 12 + inches
        else:
            return np.nan
    except:
        return np.nan

# Process each stat column
for col in stats:
    if col in df_fighters.columns:
        # Check if column is numeric
        if df_fighters[col].dtype in ['int64', 'float64']:
            numeric_stats.append(col)
        elif col == 'Height:':
            # Convert Height: from "5' 11"" format to total inches
            print(f"Converting {col} from feet/inches to total inches...")
            df_fighters[col + '_inches'] = df_fighters[col].apply(height_to_inches)
            numeric_stats.append(col + '_inches')
            # Drop the original Height: column from stats
            if col in numeric_stats:
                numeric_stats.remove(col)
        else:
            print(f"Skipping non-numeric column: {col}")

print(f"Using {len(numeric_stats)} numeric stats columns for training")

# Calculate differences for each numeric stat
for col in numeric_stats:
    def calc_diff(row):
        try:
            red_val = df_fighters.loc[row["Red"], col]
            blue_val = df_fighters.loc[row["Blue"], col]
            # Handle Series (if loc returns multiple values, take first)
            if isinstance(red_val, pd.Series):
                red_val = red_val.iloc[0]
            if isinstance(blue_val, pd.Series):
                blue_val = blue_val.iloc[0]
            # Convert to float and check for NaN
            red_val = float(red_val) if pd.notna(red_val) else 0.0
            blue_val = float(blue_val) if pd.notna(blue_val) else 0.0
            return red_val - blue_val
        except (KeyError, IndexError, ValueError, TypeError):
            return 0.0
    
    df_matches[col] = df_matches.apply(calc_diff, axis=1)
    
# drop fighter1 and fighter2 columns
df_matches.drop(["Red", "Blue"], axis=1, inplace=True)

Converting Height: from feet/inches to total inches...
Using 14 numeric stats columns for training


In [67]:
df_matches.head()

,result,Weight:,Age,SLpM:,Str. Acc.:,SApM:,Str. Def:,TD Avg.:,TD Acc.:,TD Def.:,Sub. Avg.:,Reach:,Win Percentage,Total Fights,Height:_inches
0,0,0.0,1.2,0.50,-0.15,0.17,-0.14,0.25,0.45,-0.36,0.8,0.0,-0.11,-3.0,4.0
1,0,0.0,13.3,-2.03,-0.04,-1.24,0.03,-0.53,0.05,-0.18,0.2,6.0,-0.23,3.0,5.0
2,0,0.0,11.8,1.47,-0.28,1.26,-0.01,-5.77,-0.85,-0.48,0.2,-1.0,-0.07,-11.0,-2.0
3,1,0.0,-0.9,0.40,-0.01,-1.45,0.01,-0.13,-0.08,-0.25,0.9,2.0,0.16,-2.0,2.0
4,0,-19.0,-2.4,3.26,-0.03,2.41,0.00,-1.50,0.29,0.80,0.2,6.0,0.05,13.0,3.0


In [68]:
print(len(df_matches.loc[df_matches["result"]==1]))
print(len(df_matches.loc[df_matches["result"]==0]))

5167
3070


In [69]:
df_matches.to_csv(r'training_model.csv', index=False)

In [70]:
X, y = df_matches.iloc[:, 1:], df_matches.iloc[:, 0]

# Check for and handle NaN values
print(f"Shape before cleaning: {X.shape}")
print(f"NaN values per column:\n{X.isna().sum()}")

# Fill NaN values with 0 (or median if preferred)
X = X.fillna(0)

# Ensure all columns are numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)

print(f"Shape after cleaning: {X.shape}")
print(f"Data types:\n{X.dtypes}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Test set shape: {X_test_scaled.shape}")

# Save the scaler for later use in predictions
import pickle
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved to scaler.pkl")

Shape before cleaning: (8237, 14)
NaN values per column:
Weight:           0
Age               0
SLpM:             0
Str. Acc.:        0
SApM:             0
Str. Def:         0
TD Avg.:          0
TD Acc.:          0
TD Def.:          0
Sub. Avg.:        0
Reach:            0
Win Percentage    0
Total Fights      0
Height:_inches    0
dtype: int64
Shape after cleaning: (8237, 14)
Data types:
Weight:           float64
Age               float64
SLpM:             float64
Str. Acc.:        float64
SApM:             float64
Str. Def:         float64
TD Avg.:          float64
TD Acc.:          float64
TD Def.:          float64
Sub. Avg.:        float64
Reach:            float64
Win Percentage    float64
Total Fights      float64
Height:_inches    float64
dtype: object
Training set shape: (5765, 14)
Test set shape: (2472, 14)
Scaler saved to scaler.pkl


In [71]:
# Build the neural network model
model = Sequential()

model.add(Dense(16, input_dim=X_train_scaled.shape[1],
                activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(32, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(16, activation='relu', kernel_regularizer=regularizers.L2(0.01)))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(
    loss='binary_crossentropy', 
    optimizer='adam',  # lowercase 'adam' for TensorFlow 2.x
    metrics=['binary_accuracy']
)

# Display model summary
print("Model Architecture:")
model.summary()

# Train the model with early stopping to prevent overfitting
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Split training data for validation
X_train_fit, X_val, y_train_fit, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

# Train the model
print("\nTraining model...")
history = model.fit(
    x=X_train_fit, 
    y=y_train_fit,
    validation_data=(X_val, y_val),
    epochs=200, 
    batch_size=64, 
    verbose=1,
    callbacks=[early_stopping]
)

# Evaluate on test set
test_results = model.evaluate(x=X_test_scaled, y=y_test, verbose=0)
print(f"\nTest Accuracy = {test_results[1]:.4f} ({test_results[1]*100:.2f}%)")
print(f"Test Loss = {test_results[0]:.4f}")

# Save the model
model.save('ufc_prediction_model.h5')
print("\nModel saved as 'ufc_prediction_model.h5'")

Model Architecture:


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,609 (25.82 KB)

 Trainable params: 6,609 (25.82 KB)

 Non-trainable params: 0 (0.00 B)


Training model...
Epoch 1/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - binary_accuracy: 0.6379 - loss: 2.5146 - val_binary_accuracy: 0.6201 - val_loss: 1.7453
Epoch 2/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 730us/step - binary_accuracy: 0.6344 - loss: 1.5748 - val_binary_accuracy: 0.6201 - val_loss: 1.1904
Epoch 3/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 703us/step - binary_accuracy: 0.6344 - loss: 1.1014 - val_binary_accuracy: 0.6201 - val_loss: 0.9158
Epoch 4/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - binary_accuracy: 0.6344 - loss: 0.8683 - val_binary_accuracy: 0.6201 - val_loss: 0.7761
Epoch 5/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 780us/step - binary_accuracy: 0.6361 - loss: 0.7516 - val_binary_accuracy: 0.6288 - val_loss: 0.7101
Epoch 6/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 723us/step - binary_accuracy: 0.6693 - loss: 0.6985 - val_binary_accuracy: 0.6539 - val_loss: 0.6829
Epoch 7/200
73/73 ━━━━━━━━━━━━━━━━━━━━ 0s 736us/step - binary_accuracy: 0.6837 - loss: 0.6756 - val_binary_accuracy: 0.6635 - v


Test Accuracy = 0.6780 (67.80%)
Test Loss = 0.6397

Model saved as 'ufc_prediction_model.h5'


In [72]:
def predict(model, scaler, df_fighters, Red, Blue):
    """Predict the probability of Red fighter winning against Blue fighter."""
    # Get only numeric stats (same as used in training)
    stats = [col for col in df_fighters.columns if df_fighters[col].dtype in ['int64', 'float64']]
    
    # Handle Height: conversion if needed
    if 'Height:' in df_fighters.columns:
        # Convert Height: to inches if not already converted
        if 'Height:_inches' not in df_fighters.columns:
            def height_to_inches(height_str):
                if pd.isna(height_str) or height_str == '':
                    return np.nan
                try:
                    parts = str(height_str).replace('"', '').split("'")
                    if len(parts) == 2:
                        feet = float(parts[0].strip())
                        inches = float(parts[1].strip())
                        return feet * 12 + inches
                    return np.nan
                except:
                    return np.nan
            df_fighters['Height:_inches'] = df_fighters['Height:'].apply(height_to_inches)
        if 'Height:_inches' in df_fighters.columns and 'Height:_inches' not in stats:
            stats.append('Height:_inches')
        if 'Height:' in stats:
            stats.remove('Height:')
    
    data1 = df_fighters.loc[Red, stats]
    data2 = df_fighters.loc[Blue, stats]
    data_diff = (data1 - data2).values.reshape(1, -1)
    data_diff_scaled = scaler.transform(data_diff)
    prediction = model.predict(data_diff_scaled, verbose=0)
    return prediction[0][0]

# Example predictions
try:
    if len(df_fighters) > 0:
        # Try some example predictions (adjust fighter names as needed)
        sample_fighters = df_fighters.index.tolist()[:10]
        if len(sample_fighters) >= 2:
            fighter1 = sample_fighters[0]
            fighter2 = sample_fighters[1]
            prob = predict(model, scaler, df_fighters, fighter1, fighter2)
            print(f"Prediction: {fighter1} vs {fighter2}")
            print(f"Probability of {fighter1} winning: {prob:.2%}")
except Exception as e:
    print(f"Could not run example prediction: {e}")
    print("Make sure you have fighter data loaded and the model is trained.")

Prediction: Tom Aaron vs Danny Abbadi
Probability of Tom Aaron winning: nan%


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [76]:
fight1 = predict(model, scaler, df_fighters, "Justin Gaethje", "Paddy Pimblett")
fight2 = predict(model, scaler, df_fighters, "Paddy Pimblett", "Justin Gaethje")
print(fight1)
print(fight2)

0.48978776
0.7518492


/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/griffinp/UFCMachine_Refresh/venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [77]:
fighter1_avg = (fight1 + (1 - fight2))/2
fighter1_avg


np.float32(0.3689693)

In [78]:
fighter2_avg = (fight2 + (1 - fight1))/2
fighter2_avg

np.float32(0.6310307)